# MAX-3 · Maximum Model — Single-Fold Test (Mendeley)

Trains the maximum configuration on the Mendeley fold: CORN ordinal head, full
backbone fine-tuning, 384×384 resolution, soft ordinal targets, source-reliability
weighting, sampling, curriculum, and domain-adversarial training. Results are
written to scope3_max; the original pipeline is untouched. This single fold
confirms whether the maximum stack improves on the baseline before running all
four folds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU (A100).')
print('GPU:', torch.cuda.get_device_name(0))
print('Resolution:', TM.MAX_IMG_SIZE, '| Freeze:', TM.MAX_FREEZE_STAGES, '| MRKR trust:', TM.MRKR_TRUST)
manifest = TM.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Resolution: 384 | Freeze: 0 | MRKR trust: 0.4
Copied array in 40s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


## Train maximum model on Mendeley fold

In [2]:
mech = {'sampler':True,'noise_loss':True,'curriculum':True,'domain_adv':True,
        'hierarchical':True,'use_quality':True,'soft_alpha':TM.MAX_SOFT_ALPHA,
        'freeze_stages':TM.MAX_FREEZE_STAGES,'grl_lambda_max':0.5}

test_ds = 'mendeley'
tr, va, te = TM.make_splits(manifest, test_ds, seed=0)
print(f'train={len(tr):,}  val={len(va):,}  test={len(te):,}')

res = TM.run_training('max_mendeley_seed0', tr, va, te, mech, seed=0,
                      log_fn=print)

train=45,304  val=7,995  test=8,259
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 248MB/s]


  [max_mendeley_seed0] ep1/18 loss=1.128 tr=0.339 val=0.509 w1=0.731 qwk=0.573 gap=-0.169 (582s)
  [max_mendeley_seed0] ep2/18 loss=0.918 tr=0.567 val=0.529 w1=0.754 qwk=0.619 gap=+0.037 (555s)
  [max_mendeley_seed0] ep3/18 loss=1.393 tr=0.611 val=0.542 w1=0.762 qwk=0.627 gap=+0.069 (554s)
  [max_mendeley_seed0] ep4/18 loss=1.368 tr=0.637 val=0.522 w1=0.746 qwk=0.615 gap=+0.114 (555s)
  [max_mendeley_seed0] ep5/18 loss=1.330 tr=0.661 val=0.534 w1=0.758 qwk=0.620 gap=+0.127 (554s)
  [max_mendeley_seed0] ep6/18 loss=1.417 tr=0.663 val=0.573 w1=0.779 qwk=0.634 gap=+0.090 (555s)
  [max_mendeley_seed0] ep7/18 loss=1.555 tr=0.672 val=0.572 w1=0.770 qwk=0.599 gap=+0.100 (555s)
  [max_mendeley_seed0] ep8/18 loss=1.624 tr=0.670 val=0.581 w1=0.776 qwk=0.622 gap=+0.089 (554s)
  [max_mendeley_seed0] ep9/18 loss=1.665 tr=0.683 val=0.573 w1=0.765 qwk=0.603 gap=+0.110 (553s)
  [max_mendeley_seed0] ep10/18 loss=1.695 tr=0.680 val=0.590 w1=0.785 qwk=0.633 gap=+0.090 (553s)
  [max_mendeley_seed0] ep11/1

## Compare to baseline (original H, Mendeley fold)

In [3]:
import json, glob
def load(p):
    try: return json.load(open(str(p)))
    except Exception: return None

h = load(config.RESULTS_DIR/'fold1_test_mendeley_seed0.json')
h_w1 = np.nan
zf = sorted(glob.glob(str(config.RESULTS_DIR/'fold1_test_mendeley_seed*_preds.npz')))
if zf:
    z=np.load(zf[0]); h_w1=float((np.abs(z['y_true']-z['y_pred'])<=1).mean())

print('Mendeley fold — baseline H vs MAX (seed 0):')
print(f"{'metric':12s}{'H (224)':>12s}{'MAX (384)':>12s}")
if h:
    print(f"{'exact acc':12s}{h['external_acc5']:>12.3f}{res['external_acc5']:>12.3f}")
    print(f"{'within-1':12s}{h_w1:>12.3f}{res['external_within1']:>12.3f}")
    print(f"{'QWK':12s}{h['external_qwk']:>12.3f}{res['external_qwk']:>12.3f}")
    print(f"{'gap (pp)':12s}{h['gap']*100:>12.1f}{res['gap']*100:>12.1f}")
    print()
    print(f"Delta exact: {res['external_acc5']-h['external_acc5']:+.3f}")
    print(f"Delta within-1: {res['external_within1']-h_w1:+.3f}")
else:
    print(f"MAX: exact={res['external_acc5']:.3f} within1={res['external_within1']:.3f} qwk={res['external_qwk']:.3f} gap={res['gap']*100:.1f}pp")

Mendeley fold — baseline H vs MAX (seed 0):
metric           H (224)   MAX (384)
exact acc          0.338       0.320
within-1           0.584       0.615
QWK                0.354       0.320
gap (pp)            20.2        28.7

Delta exact: -0.018
Delta within-1: +0.032
